# Cost Functions and Unit Selection

From the target phonemes, a path needs to be found through the state transition network (sequence of database units), that minimizes a cost function.

![image.png](resources/UnitSelectionCosts.png)

## Target Cost

The target cost is the weighted sum of the differences between elements of the target and candidate feature vectors.

$$ C^{t}(t_i, u_i) = \sum_{j=1}^{p} w_j C^{t}_{j}(t_i, u_i) $$

Where $p$ ranges from 20 to 30 and is the number of sub-costs, or differences between the feature vectors.

## Concatenation Cost

Determined by the weight of $q$ concatenation sub-costs.

$$ C^c_{(u_{i-1}, u_i)} = \sum_{j=1}^q w^c_j C_j^{c}(u_{i-1}, u_i) $$

## Final Cost Function

$$ C(t_1^n, u_1^n) = \sum_{i=1}^{n} \sum_{j=1}^{p} w_j^tC_j^t(t_i, u_i^t) + \sum_{i=2}^{n} \sum_{j=1}^{q} w_j^cC_j^c(u_j, u_i) + C^s(S, u_1) + C^c(u_n, S) $$

Where $S$ is silence.

The units are then selected in order to minimize this cost function via Viterbi search.

$$ \bar{u}_1^n = \min_{u_1,\ldots,u_n} \mathcal{C}(t_1^n,u_1^n) $$

# Feature Vectors

The paper cites that 20-30 features were used for calculating the target cost. The paper cites several features of the phonemes: pitch, power, and duration. The paper also looks at prosodic characteristics of the preceding and following phonemes.

Calculation of many of the feature vectors is quite involved, so subsets of feature vectors should be established. Additional features can be added and calculated as the experiment progresses.

For the moment discarding all temporal features from the speech dataset. There remain features describing phonemes: as vowels or consonants, voicing, articulation manner, articulation place, pre/post silence markers. We'll also consider the proceeding and following phonemes and their duration.

## Feature Vector Shape

Features that comprise the vector and their shape.
- Vowel or consonant (binary option).
- Voicing (phoneme is voiced or not) (binary option).
- Articulation manner (plosive, fricative, affricate, nasal, liquid, glide, vowel, silence) (one-hot).
- Articulation place (bilabial, labiodental, dental, alveolar, postalveolar, palatal, velar, glottal, none) (one-hot).
- Proceeding phoneme (one-hot).
- Current phoneme (one-hot).
- Following phoneme (one-hot).
- Duration (float).

The first 4 features are known beforehand from a lookup table. The last 4 features can be read from each `lab/` file.

In [ ]:
# Articulatory/Phonological mapping table for ARPAbet phonemes
# Categories:
# - is_vowel: 1 if vowel, 0 otherwise [cite: 39]
# - is_voiced: 1 if vocal cords vibrate, 0 otherwise [cite: 40]
# - manner: plosive, fricative, affricate, nasal, liquid, glide, vowel, silence [cite: 41]
# - place: bilabial, labiodental, dental, alveolar, postalveolar, palatal, velar, glottal, none [cite: 42]

# Vocabulary lists for categorical fields
MANNERS = ['vowel', 'plosive', 'fricative', 'affricate', 'nasal', 'liquid', 'glide', 'silence']
PLACES = ['bilabial', 'labiodental', 'dental', 'alveolar', 'postalveolar', 'palatal', 'velar', 'glottal', 'none']

PHONEME_FEATURES = {
    # Silence / Pauses
    'pau': {'is_vowel': 0, 'is_voiced': 0, 'manner': 'silence', 'place': 'none'},
    'ssil': {'is_vowel': 0, 'is_voiced': 0, 'manner': 'silence', 'place': 'none'},
    
    # Vowels (All vowels are voiced)
    'aa': {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'ae': {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'ah': {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'ao': {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'aw': {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'ax': {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'ay': {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'eh': {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'er': {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'ey': {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'ih': {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'iy': {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'ow': {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'oy': {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'uh': {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'uw': {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},

    # Stops / Plosives
    'p':  {'is_vowel': 0, 'is_voiced': 0, 'manner': 'plosive', 'place': 'bilabial'},
    'b':  {'is_vowel': 0, 'is_voiced': 1, 'manner': 'plosive', 'place': 'bilabial'},
    't':  {'is_vowel': 0, 'is_voiced': 0, 'manner': 'plosive', 'place': 'alveolar'},
    'd':  {'is_vowel': 0, 'is_voiced': 1, 'manner': 'plosive', 'place': 'alveolar'},
    'k':  {'is_vowel': 0, 'is_voiced': 0, 'manner': 'plosive', 'place': 'velar'},
    'g':  {'is_vowel': 0, 'is_voiced': 1, 'manner': 'plosive', 'place': 'velar'},

    # Fricatives
    'f':  {'is_vowel': 0, 'is_voiced': 0, 'manner': 'fricative', 'place': 'labiodental'},
    'v':  {'is_vowel': 0, 'is_voiced': 1, 'manner': 'fricative', 'place': 'labiodental'},
    'th': {'is_vowel': 0, 'is_voiced': 0, 'manner': 'fricative', 'place': 'dental'},
    'dh': {'is_vowel': 0, 'is_voiced': 1, 'manner': 'fricative', 'place': 'dental'},
    's':  {'is_vowel': 0, 'is_voiced': 0, 'manner': 'fricative', 'place': 'alveolar'},
    'z':  {'is_vowel': 0, 'is_voiced': 1, 'manner': 'fricative', 'place': 'alveolar'},
    'sh': {'is_vowel': 0, 'is_voiced': 0, 'manner': 'fricative', 'place': 'postalveolar'},
    'zh': {'is_vowel': 0, 'is_voiced': 1, 'manner': 'fricative', 'place': 'postalveolar'},
    'hh': {'is_vowel': 0, 'is_voiced': 0, 'manner': 'fricative', 'place': 'glottal'},

    # Affricates
    'ch': {'is_vowel': 0, 'is_voiced': 0, 'manner': 'affricate', 'place': 'postalveolar'},
    'jh': {'is_vowel': 0, 'is_voiced': 1, 'manner': 'affricate', 'place': 'postalveolar'},

    # Nasals
    'm':  {'is_vowel': 0, 'is_voiced': 1, 'manner': 'nasal', 'place': 'bilabial'},
    'n':  {'is_vowel': 0, 'is_voiced': 1, 'manner': 'nasal', 'place': 'alveolar'},
    'ng': {'is_vowel': 0, 'is_voiced': 1, 'manner': 'nasal', 'place': 'velar'},

    # Liquids & Glides
    'l':  {'is_vowel': 0, 'is_voiced': 1, 'manner': 'liquid', 'place': 'alveolar'},
    'r':  {'is_vowel': 0, 'is_voiced': 1, 'manner': 'liquid', 'place': 'alveolar'},
    'w':  {'is_vowel': 0, 'is_voiced': 1, 'manner': 'glide', 'place': 'bilabial'},
    'y':  {'is_vowel': 0, 'is_voiced': 1, 'manner': 'glide', 'place': 'palatal'},
}

In [ ]:
# For each file in "datasets/cmu_us_awb_arctic/lab"
# For each line in file 2:N-2 (1-indexed)
# Extract current phoneme, next phoneme, previous phoneme, duration.